In [ ]:
import importlib
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.signal

import utils
from utils import bpsk_acquisition, collect_metadata_utils, sample_streaming, tracking_channel
from utils.signal_interfaces import (
    SignalFamily,
    build_signal_definitions,
    create_tracking_channels,
)


plt.rcParams.update({'font.size': 16})

In [ ]:
# Note: you will need to change the filepath to point to where your data is stored
#  I have mine stored in a "local-data" folder within the project directory
local_data_dir = Path(utils.__file__).parent.parent / "local-data"
collects_dir = local_data_dir / "collects"
available_experiment_names = sorted(map(lambda fp: fp.name, collects_dir.iterdir()))
print("Available experiments:", ", ".join([str(name) for name in available_experiment_names]))
experiment_name = available_experiment_names[2]
experiment_dir = collects_dir / experiment_name
# Note: I keep a metadata.yml file in each experiment directory to keep track of sample metadata
#  and what are the available collect filenames.  You can either create your own metadata.yml,
#  or modify the code below to directly chose your data filepath and set sample parameters.
metadata_filepath = experiment_dir / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

collect_id = metadata.collect_ids[2]
band_id = metadata.band_ids[0]

collect_config = metadata.collects[collect_id]
channel_id = collect_config.channel_config_id
band_config = metadata.band_configurations[band_id]
channel_config = metadata.channel_configurations[channel_id]

inter_freq_l1_hz = band_config.inter_freq
samp_rate = channel_config.samp_rate
sample_params = channel_config.sample_params
collect_filepath = experiment_dir / collect_config.filename

In [ ]:
# Load acquisition results from file
acq_results_directory = local_data_dir / "acquisition-results"
acq_results_directory.mkdir(parents=True, exist_ok=True)
acq_results_version_id = "v1"
acq_results_filepath = acq_results_directory / f"{collect_id}.{acq_results_version_id}.pkl"
with open(acq_results_filepath, "rb") as f:
    acq_results: dict[str, bpsk_acquisition.AcquisitionResult] = pickle.load(f)

acquired_signal_ids = sorted(list(filter(
    lambda sig_id: acq_results[sig_id].signal_detected, acq_results.keys()
)))
print(f"Acquired signals in collect {collect_id}: ")
print(", ".join(acquired_signal_ids))

In [ ]:
buffer_duration_ms = 40
block_duration_ms = 1
buffer_size_samples = int(samp_rate * buffer_duration_ms / 1e3)
block_size_samples = int(samp_rate * block_duration_ms / 1e3)

# duration_to_track_ms = 60000
duration_to_track_ms = 10000
# duration_to_track_ms = 2000
num_blocks_to_track = duration_to_track_ms // block_duration_ms
num_buffers_to_process = duration_to_track_ms // buffer_duration_ms

tracking_signal_ids = acquired_signal_ids
# tracking_signal_ids = ["G01", "G17", "G30"]

# The signal's code, how it occupies the chip clock, and which component drives
# which loop all come from the shared definitions now, instead of being spelled
# out per notebook.  See utils/code_components.py and utils/signal_interfaces.py.
signal_definitions = build_signal_definitions(SignalFamily.L1CA)

# Use same tracking loop parameters for all signals
tracking_loop_params = tracking_channel.TrackingLoopParameters(
    DLL_bandwidth_hz=2.0,
    PLL_bandwidth_hz=20.0,
    FLL_bandwidth_hz=50.0,
    nominal_update_period_ms=block_duration_ms,
    corr_period_ms=block_duration_ms,
    EPL_chip_spacing=0.5
)

# Seeds each channel from its acquisition result (Doppler, code phase, epoch).
tracking_channels = create_tracking_channels(
    signal_definitions=signal_definitions,
    acquisition_results=acq_results,
    tracking_signal_ids=tracking_signal_ids,
    loop_params=tracking_loop_params,
    output_capacity=num_blocks_to_track,
)

In [ ]:
with sample_streaming.FileSampleStream(
        collect_filepath,
        sample_params,
        buffer_size_samples,
    ) as sample_stream:

    sample_buffer_generator = sample_stream.sample_buffer_generator()
    
    for i_buffer, buffer_samples in enumerate(sample_buffer_generator):
        if i_buffer >= num_buffers_to_process:
            break
        uptime_ms = i_buffer * buffer_duration_ms
        print(f"\rProcessing buffer {i_buffer} (uptime {uptime_ms} ms)...", end="")
        
        # Mixdown to baseband
        # phi_IF = 2 * np.pi * inter_freq_l1_hz * (uptime_seconds + tracking_loop_params.block_sample_time_arr)
        # sample_block *= np.exp(-1j * phi_IF)
        mixdown_phi0 = inter_freq_l1_hz * (uptime_ms * 1e-3)
        sample_streaming.mixdown_samples(buffer_samples, buffer_samples, samp_rate, mixdown_phi0, inter_freq_l1_hz)
        
        # Track each signal
        sample_buffer = sample_streaming.SampleBuffer(buffer_samples, uptime_ms, samp_rate)
        for sig_id, channel in tracking_channels.items():
            channel.process_sample_buffer(sample_buffer)

In [ ]:
# Save tracking results to file
tracking_results_directory = os.path.join(local_data_dir, "tracking-results")
os.makedirs(tracking_results_directory, exist_ok=True)
tracking_results_version_id = "v1"
tracking_results_filepath = os.path.join(
    tracking_results_directory, f"{collect_id}.{tracking_results_version_id}.pkl"
)
all_tracking_outputs = {
    sig_id: channel.outputs for sig_id, channel in tracking_channels.items()
}
with open(tracking_results_filepath, "wb") as f:
    pickle.dump(tracking_loop_params, f)
    pickle.dump(all_tracking_outputs, f)

In [ ]:
print("\nTracking complete. Tracking results saved to:", tracking_results_filepath)